#read table from silver layer

In [0]:
df = spark.read.format("delta").table("`ttc-lakehouse`.silver.stops")

##select relevent attributes for building the gold stops dimension

In [0]:
df = df.select('stop_id','stop_name','stop_lat','stop_lon','wheelchair_boarding')

## Enforce geographic coordinates data types for analytical and spatial operations

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("stop_lat", col("stop_lat").cast("double")) \
       .withColumn("stop_lon", col("stop_lon").cast("double")) \
       .withColumn("wheelchair_boarding", col("wheelchair_boarding").cast("int"))

## Add human-readable accessibility description based on GTFS wheelchair_boarding codes

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "wheelchair_access_desc",
    when(col("wheelchair_boarding") == 1, "Accessible")
    .when(col("wheelchair_boarding") == 2, "Not Accessible")
    .when(col("wheelchair_boarding") == 0, "Unknown")
)

##write into Gold layer

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("`ttc-lakehouse`.gold.dim_stops")